# Red ER no lineal: el entrelazamiento pesa más donde la gravedad es débil

**Proyecto:** *Vacuum Particle Journey*, tercera parte.

**Qué aprendimos en el notebook 2:**
- Con κ y λ constantes y universales, la red ER generaliza (validación cruzada estable), supera a Verlinde y da **κ ≈ 5.6, muy cerca de la proporción cósmica Ω_c/Ω_b ≈ 5.36**.
- Pero queda muy lejos de ΛCDM (ΔBIC ≈ 88 000), fuerza masas estelares demasiado bajas (Υ ≈ 0.22) y no puede explicar que las galaxias enanas sean mucho más "oscuras" que las grandes.

**Hipótesis nueva:** el número de puentes ER por región depende de la gravedad local. Donde la materia es densa y caliente, las interacciones rompen el entrelazamiento (**decoherencia**). En las afueras, frías y difusas, la red sobrevive. Lo escribimos con la aceleración bariónica local $g_b = v_b^2/r$:

$$
\rho_{ER}(\mathbf{x}) = \kappa_0\int \Big(\frac{a_\star}{g_b(\mathbf{y})}\Big)^{\beta}\rho_b(\mathbf{y})\,K_\lambda(|\mathbf{x}-\mathbf{y}|)\,d^3y
$$

Todos los parámetros ($\kappa_0, \beta, \lambda$) son **universales**; $a_\star = cH_0/6$ es solo la escala de referencia. Con $\beta = 0$ se recupera el modelo de κ constante. **Con $\beta = 1/2$ salen curvas planas y $v^4 \propto M_b$ (Tully–Fisher)**: esa es la predicción a comprobar.

**Modelos que se comparan:**

| Modelo | Parámetros por galaxia | Parámetros globales |
|---|---|---|
| Solo bariones | Υ | – |
| NFW libre (referencia de ajuste) | Υ, V200, c | – |
| NFW ΛCDM (relación c–M) | Υ, V200 | – |
| Verlinde | Υ | – (a₀ = cH₀) |
| **RAR de McGaugh** (ley empírica, tipo MOND) | Υ | – (g† = 1.2·10⁻¹⁰ m/s²) |
| Red ER κ constante (notebook 2) | Υ | κ, λ |
| **Red ER κ = Ω_c/Ω_b** (predicción pura) | Υ | λ |
| **Red ER no lineal κ(g_b)** | Υ | κ₀, β, λ |

La **RAR de McGaugh** es el rival clave: una fórmula empírica que ajusta muy bien las curvas de rotación sin materia oscura. Si la red ER no lineal empata con ella, la novedad está en la *explicación física* y en lo que la red predice de distinto (el alcance λ). Si la supera, mejor todavía.

> Tiempo en Colab: ~10–20 min con la muestra completa. `QUICK = True` para 40 galaxias.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {
    "Datos": "#222222",
    "Solo bariones": "#8a8a85",
    "NFW libre": "#2a78d6",
    "NFW ΛCDM (c–M)": "#6250d6",
    "Verlinde": "#eb6834",
    "RAR McGaugh": "#e87ba4",
    "ER κ constante": "#eda100",
    "ER κ = Ωc/Ωb": "#008300",
    "ER no lineal κ(g)": "#1baf7a",
}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6                 # kpc (km/s)^2 / Msun
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
A0 = C_KMS * H0_KPC            # c*H0 en (km/s)^2/kpc
A_STAR = A0 / 6                # escala de referencia (≈1.13e-10 m/s^2)
G_DAGGER = 1.2e-10 * KPC_M / 1e6   # escala de McGaugh en (km/s)^2/kpc
OMEGA_RATIO = 0.1200 / 0.02237     # Ω_c h² / Ω_b h² (Planck 2018) ≈ 5.36
TO_MS2 = 1e6 / KPC_M           # (km/s)^2/kpc -> m/s^2
print(f"a★ = {A_STAR*TO_MS2:.2e} m/s² | g† = 1.2e-10 m/s² | Ω_c/Ω_b = {OMEGA_RATIO:.2f}")

## 1. Datos SPARC (igual que en el notebook 2)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

## 2. Modelos

**Red ER (una sola implementación para las tres variantes).** En cada galaxia se reparte la masa bariónica en cascarones. Cada cascarón se pondera con $w = (a_\star/g_b)^\beta$, donde $g_b$ es la aceleración bariónica en ese radio, y se reparte con el núcleo de alcance $\lambda$. Para que el ajuste global sea rápido, la parte que solo depende de $\lambda$ (el núcleo y la integral de masa) se guarda como una matriz $T_\lambda$:
$$v^2_{ER}(r_i) = \kappa_0 \sum_j T_\lambda[i,j]\; w_j\,\Delta(GM_b)_j$$
El peso se limita a $w \le 10^4$ para evitar divisiones por cero en el centro, donde $g_b \to 0$.

**RAR de McGaugh (2016):** $\;g_{obs} = \dfrac{g_b}{1 - e^{-\sqrt{g_b/g_\dagger}}}$, con $v^2 = r\,g_{obs}$.

In [ ]:
def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

def prior(u):
    return np.log10(u / 0.5) / 0.1

def chi2_on(g, v, mask=None):
    res = ((g["vobs"] - v) / g["err"]) ** 2
    return float(res.sum() if mask is None else res[mask].sum())

# ---------------- Red ER ----------------
_CACHE = {}
def er_setup(g, lam, n_grid=120):
    key = (g["name"], round(float(lam), 8), len(g["r"]))
    if key in _CACHE:
        return _CACHE[key]
    if len(_CACHE) > 4000:
        _CACHE.clear()
    r = g["r"]
    rg = np.linspace(r.max() / n_grid, r.max(), n_grid)
    S, R = rg[:, None], rg[None, :]
    kern = (np.exp(-np.abs(S - R) / lam) - np.exp(-(S + R) / lam)) / (8 * np.pi * lam * S * R)
    Mc = cumulative_trapezoid(4 * np.pi * rg[:, None] ** 2 * kern, rg, axis=0, initial=0.0)
    eye = np.eye(n_grid)
    P = np.array([np.interp(r, np.r_[0.0, rg], np.r_[0.0, e]) for e in eye]).T
    s = dict(rg=rg, T=(P @ Mc) / r[:, None])
    for comp in ("vgas2", "vstar2"):
        GMg = np.interp(rg, np.r_[0.0, r], np.r_[0.0, r * g[comp]])
        s["d_" + comp] = np.diff(np.r_[0.0, GMg])
        s["g_" + comp] = np.interp(rg, r, g[comp])
    _CACHE[key] = s
    return s

def er_E(s, U, betas):
    # v²_ER por unidad de κ0 -> forma (nβ, nΥ, N)
    dGM = s["d_vgas2"][None, :] + U[:, None] * s["d_vstar2"][None, :]
    gb = np.clip(s["g_vgas2"][None, :] + U[:, None] * s["g_vstar2"][None, :], 1e-2, None) / s["rg"][None, :]
    lw = np.log(A_STAR / gb)
    W = np.exp(np.minimum(betas[:, None, None] * lw[None], np.log(1e4)))
    return np.einsum("in,bun->bui", s["T"], W * dGM[None])

def er_v2(g, u, kap, beta, lam):
    E = er_E(er_setup(g, lam), np.array([u]), np.array([beta]))[0, 0]
    return vb2(g, u) + kap * E, kap * E

# ---------------- Otros modelos ----------------
def v2_verlinde_extra(r, v2b, smooth_dex=0.08):
    GM = np.maximum.accumulate(r * np.clip(v2b, 0, None))
    grid = np.linspace(np.log10(r.min()), np.log10(r.max()), 200)
    GMg = np.interp(grid, np.log10(r), GM)
    dx = max(grid[1] - grid[0], 1e-6)
    n = max(int(3 * smooth_dex / dx), 1)
    k = np.exp(-0.5 * (np.arange(-n, n + 1) * dx / smooth_dex) ** 2); k /= k.sum()
    GMs = np.convolve(np.pad(GMg, n, mode="edge"), k, mode="valid")
    rgrid = 10 ** grid
    dy = np.gradient(rgrid * GMs, rgrid)
    return np.interp(r, rgrid, np.sqrt(A0 / 6 * np.clip(dy, 0, None)))

def c_dutton(V200):
    M200 = V200**3 / (10 * G * H0_KPC)
    return 10 ** (0.905 - 0.101 * np.log10(M200 / (1e12 / (H0 / 100))))

def v2_rar(g, u):
    gb = np.clip(vb2(g, u), 1e-6, None) / g["r"]
    return g["r"] * gb / (1 - np.exp(-np.sqrt(gb / G_DAGGER)))

LS_MODELS = {
    "Solo bariones":  dict(names=["Υ"], p0=[0.5], lo=[0.1], hi=[1.5], v2=lambda g, p: vb2(g, p[0])),
    "NFW libre":      dict(names=["Υ", "V200", "c"], p0=[0.5, 100, 10], lo=[0.1, 10, 1], hi=[1.5, 500, 50],
                           v2=lambda g, p: vb2(g, p[0]) + _v2_nfw(g["r"], p[1], p[2])),
    "NFW ΛCDM (c–M)": dict(names=["Υ", "V200"], p0=[0.5, 100], lo=[0.1, 10], hi=[1.5, 500],
                           v2=lambda g, p: vb2(g, p[0]) + _v2_nfw(g["r"], p[1], c_dutton(p[1]))),
    "Verlinde":       dict(names=["Υ"], p0=[0.5], lo=[0.1], hi=[1.5],
                           v2=lambda g, p: vb2(g, p[0]) + v2_verlinde_extra(g["r"], vb2(g, p[0]))),
    "RAR McGaugh":    dict(names=["Υ"], p0=[0.5], lo=[0.1], hi=[1.5], v2=lambda g, p: v2_rar(g, p[0])),
}
ER_MODELS = ["ER κ constante", "ER κ = Ωc/Ωb", "ER no lineal κ(g)"]
N_SHARED = {"ER κ constante": 2, "ER κ = Ωc/Ωb": 1, "ER no lineal κ(g)": 3}
ORDER = list(LS_MODELS) + ER_MODELS

# Comprobaciones rápidas
_g = next(iter(GAL.values()))
_s = er_setup(_g, 0.3)
_E0 = er_E(_s, np.array([0.5]), np.array([0.0]))[0, 0]
print("Test β=0, λ pequeño: v²_ER/v_b² ≈", np.round(np.median(_E0[len(_E0)//2:] / np.clip(vb2(_g, 0.5)[len(_E0)//2:], 1e-3, None)), 2),
      "(debería ser ≈ 1)")

## 3. Ajuste por mínimos cuadrados (modelos con parámetros por galaxia)

In [ ]:
def fit_ls(g, name, mask=None, n_starts=4, rng=RNG):
    m = LS_MODELS[name]
    lo, hi = np.array(m["lo"], float), np.array(m["hi"], float)
    sel = np.ones(len(g["r"]), bool) if mask is None else mask
    def res(p):
        v = np.sqrt(np.clip(m["v2"](g, p), 0, None))
        return np.r_[((g["vobs"] - v) / g["err"])[sel], prior(p[0])]
    starts = [np.clip(np.array(m["p0"], float), lo, hi)] + \
             [lo + rng.random(len(lo)) * (hi - lo) for _ in range(n_starts - 1 if len(lo) > 1 else 0)]
    best = None
    for p0 in starts:
        try:
            r = least_squares(res, p0, bounds=(lo, hi), method="trf")
            if best is None or r.cost < best.cost:
                best = r
        except Exception:
            pass
    v = np.sqrt(np.clip(m["v2"](g, best.x), 0, None))
    return dict(p=best.x, v=v, chi2=chi2_on(g, v, sel), n=int(sel.sum()), k=len(best.x))

def fit_er_one(g, kap, beta, lam, mask=None):
    sel = np.ones(len(g["r"]), bool) if mask is None else mask
    def f(lu):
        v2, _ = er_v2(g, 10 ** lu, kap, beta, lam)
        return chi2_on(g, np.sqrt(np.clip(v2, 0, None)), sel) + prior(10 ** lu) ** 2
    r = minimize_scalar(f, bounds=(-1, np.log10(1.5)), method="bounded", options=dict(xatol=1e-3))
    u = 10 ** r.x
    v2, v2er = er_v2(g, u, kap, beta, lam)
    v = np.sqrt(np.clip(v2, 0, None))
    ratio = v2er[-1] / max(vb2(g, u)[-1], 1e-3)       # M_ER / M_b dentro del último radio
    return dict(p=np.array([u]), v=v, chi2=chi2_on(g, v, sel), n=int(sel.sum()), k=1, ratio=ratio)

## 4. Ajustes globales de las tres variantes de la red ER

Mismo método que en el notebook 2: rejilla gruesa, rejilla fina y, para cada combinación global, el mejor Υ en cada galaxia. La incertidumbre es la región $\Delta\chi^2 \le \Delta_{crit}\cdot s$, con $\Delta_{crit}$ = 1.00, 2.30 o 3.53 para 1, 2 o 3 parámetros globales.

- **κ constante:** β = 0; se ajustan κ y λ.
- **κ = Ω_c/Ω_b:** β = 0 y κ = 5.36 fijo; solo se ajusta λ. Si el ajuste apenas empeora respecto a κ libre, la coincidencia cósmica pasa a ser una **predicción**.
- **No lineal:** se ajustan κ₀, β y λ.

In [ ]:
U_GRID = np.geomspace(0.1, 1.5, 40)
PRIOR_U = prior(U_GRID) ** 2

def er_surface(galaxies, lams, kaps, betas):
    surf = np.zeros((len(lams), len(kaps), len(betas)))
    for i, lam in enumerate(lams):
        for g in galaxies.values():
            E = er_E(er_setup(g, lam), U_GRID, betas)                               # b,u,N
            base = g["vgas2"][None, :] + U_GRID[:, None] * g["vstar2"][None, :]    # u,N
            v2 = base[None, None] + kaps[:, None, None, None] * E[None]             # k,b,u,N
            c = (((g["vobs"] - np.sqrt(np.clip(v2, 0, None))) / g["err"]) ** 2).sum(-1) + PRIOR_U[None, None, :]
            surf[i] += c.min(-1)
    return surf

def global_er(galaxies, mode, verbose=True):
    t0 = time.time()
    if mode == "ER κ constante":
        lams, kaps, betas = np.geomspace(0.3, 300, 18), np.geomspace(0.01, 300, 50), np.array([0.0])
    elif mode == "ER κ = Ωc/Ωb":
        lams, kaps, betas = np.geomspace(0.3, 300, 30), np.array([OMEGA_RATIO]), np.array([0.0])
    else:
        lams, kaps, betas = np.geomspace(0.3, 300, 14), np.geomspace(0.003, 100, 32), np.linspace(0, 1.2, 13)
    s1 = er_surface(galaxies, lams, kaps, betas)
    i, j, b = np.unravel_index(np.argmin(s1), s1.shape)
    edge = []
    if len(lams) > 1 and i in (0, len(lams) - 1): edge.append(f"λ={lams[i]:.3g}")
    if len(kaps) > 1 and j in (0, len(kaps) - 1): edge.append(f"κ={kaps[j]:.3g}")
    if len(betas) > 1 and b in (0, len(betas) - 1): edge.append(f"β={betas[b]:.3g}")
    lams2 = np.unique(np.clip(lams[i] * np.geomspace(10**-0.2, 10**0.2, 9), 0.3, 300))
    kaps2 = kaps if len(kaps) == 1 else kaps[j] * np.geomspace(10**-0.25, 10**0.25, 17)
    betas2 = betas if len(betas) == 1 else np.unique(np.clip(betas[b] + np.linspace(-0.1, 0.1, 9), 0, 1.5))
    s2 = er_surface(galaxies, lams2, kaps2, betas2)
    i2, j2, b2 = np.unravel_index(np.argmin(s2), s2.shape)
    out = dict(mode=mode, lam=lams2[i2], kappa=kaps2[j2], beta=betas2[b2], chi2=s2.min(),
               fine=(lams2, kaps2, betas2, s2), coarse=(lams, kaps, betas, s1), edge=edge)
    if verbose:
        print(f"{mode}: κ₀={out['kappa']:.3g}, β={out['beta']:.3g}, λ={out['lam']:.3g} kpc "
              f"({time.time()-t0:.0f} s)" + (f"  ⚠ borde de rejilla: {', '.join(edge)}" if edge else ""))
    return out

def uncertainties(gf, n_gal):
    lams2, kaps2, betas2, s2 = gf["fine"]
    nfree = N_SHARED[gf["mode"]]
    s_fac = max(gf["chi2"] / (N_TOT - n_gal - nfree), 1.0)
    crit = {1: 1.0, 2: 2.30, 3: 3.53}[nfree] * s_fac
    L, K, B = np.meshgrid(lams2, kaps2, betas2, indexing="ij")
    ins = (s2 - s2.min()) <= crit
    rng_ = lambda X: (X[ins].min(), X[ins].max())
    out = dict(lam=rng_(L), kappa=rng_(K), beta=rng_(B), s=s_fac)
    for nm, X, grid in (("λ", L, lams2), ("κ₀", K, kaps2), ("β", B, betas2)):
        if len(grid) > 1 and (X[ins].min() <= grid.min() or X[ins].max() >= grid.max()):
            print(f"  Aviso ({gf['mode']}): el intervalo 1σ de {nm} toca el borde de la rejilla fina; el error real es mayor.")
    return out

GLOB = {m: global_er(GAL, m) for m in ER_MODELS}
rows = []
for m, gf in GLOB.items():
    u = uncertainties(gf, len(GAL))
    rows.append({"Modelo": m, "κ₀": f"{gf['kappa']:.3g} [{u['kappa'][0]:.3g}–{u['kappa'][1]:.3g}]",
                 "β": f"{gf['beta']:.3g} [{u['beta'][0]:.3g}–{u['beta'][1]:.3g}]",
                 "λ [kpc]": f"{gf['lam']:.3g} [{u['lam'][0]:.3g}–{u['lam'][1]:.3g}]",
                 "χ²/g.l.": round(gf["chi2"] / (N_TOT - len(GAL) - N_SHARED[m]), 2)})
print("\nParámetros globales (entre corchetes: intervalo 1σ, inflado por s):")
pd.DataFrame(rows).set_index("Modelo")

## 5. Todos los modelos, galaxia a galaxia

In [ ]:
t0 = time.time()
FITS, rows = {}, []
for n_i, (name, g) in enumerate(GAL.items()):
    F = {m: fit_ls(g, m) for m in LS_MODELS}
    for m in ER_MODELS:
        gf = GLOB[m]
        F[m] = fit_er_one(g, gf["kappa"], gf["beta"], gf["lam"])
    FITS[name] = F
    for m, f in F.items():
        rows.append(dict(galaxia=name, modelo=m, chi2=f["chi2"], n=f["n"], k=f["k"], ups=f["p"][0],
                         chi2r=f["chi2"] / max(f["n"] - f["k"], 1), ratio=f.get("ratio", np.nan)))
    if (n_i + 1) % 25 == 0:
        print(f"  {n_i+1}/{len(GAL)} ({time.time()-t0:.0f} s)")
RES = pd.DataFrame(rows)
print(f"Hecho en {time.time()-t0:.0f} s")

## 6. Prueba 1: BIC total

Los parámetros globales se cuentan una vez para toda la muestra. Fíjate sobre todo en tres comparaciones:
- **No lineal vs κ constante:** ¿mejora la dependencia con la aceleración?
- **κ = Ω_c/Ω_b vs κ constante:** ¿cuesta algo fijar κ al valor cósmico?
- **No lineal vs RAR y vs NFW ΛCDM:** los rivales de verdad.

In [ ]:
summ = []
for m in ORDER:
    d = RES[RES.modelo == m]
    K = int(d.k.sum() + N_SHARED.get(m, 0))
    chi2 = d.chi2.sum()
    summ.append(dict(Modelo=m, parametros=K, chi2_total=round(chi2), chi2r_mediana=round(d.chi2r.median(), 2),
                     galaxias_chi2r_menor_2=f"{np.mean(d.chi2r < 2):.0%}", Υ_mediana=round(d.ups.median(), 2),
                     BIC_total=round(chi2 + K * np.log(N_TOT))))
SUMM = pd.DataFrame(summ).set_index("Modelo")
SUMM["ΔBIC"] = SUMM.BIC_total - SUMM.BIC_total.min()
SUMM

## 7. ¿Se cumple la predicción β ≈ 1/2? ¿Y la proporción cósmica?

- Si el intervalo de β incluye 0.5, la red ER produce de forma natural curvas planas y Tully–Fisher.
- $M_{ER}/M_b$ dentro del último radio medido (mediana de la muestra) frente a Ω_c/Ω_b = 5.36. Ojo: una galaxia solo "ve" la parte interior de la red, así que se espera algo menor que el valor cósmico.

In [ ]:
gnl = GLOB["ER no lineal κ(g)"]; unl = uncertainties(gnl, len(GAL))
print(f"β = {gnl['beta']:.3f}  (1σ: {unl['beta'][0]:.3f} – {unl['beta'][1]:.3f})  ->  "
      + ("INCLUYE 0.5 ✓" if unl["beta"][0] <= 0.5 <= unl["beta"][1] else "no incluye 0.5"))
d = RES[RES.modelo == "ER no lineal κ(g)"]
print(f"M_ER/M_b en el último radio: mediana {d.ratio.median():.2f} (16–84 %: "
      f"{d.ratio.quantile(.16):.2f} – {d.ratio.quantile(.84):.2f});  Ω_c/Ω_b = {OMEGA_RATIO:.2f}")

dc, do = SUMM.loc["ER κ constante", "BIC_total"], SUMM.loc["ER κ = Ωc/Ωb", "BIC_total"]
print(f"Fijar κ = Ω_c/Ω_b cambia el BIC en {do - dc:+.0f} respecto a κ libre "
      + ("(no empeora: la coincidencia sobrevive ✓)" if do - dc < 2 else
         "(empeora poco)" if do - dc < 10 else "(empeora: la coincidencia no basta)"))

# κ efectivo local en función de g_b (curva universal del modelo)
gg = np.geomspace(1e-12, 1e-8, 100) / TO_MS2
kap_eff = gnl["kappa"] * np.minimum((A_STAR / gg) ** gnl["beta"], 1e4)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(gg * TO_MS2, kap_eff, color=COL["ER no lineal κ(g)"], label="κ(g_b) ajustado")
ax.axhline(OMEGA_RATIO, color=COL["ER κ = Ωc/Ωb"], ls="--", lw=1.5, label="Ω_c/Ω_b")
ax.axhline(GLOB["ER κ constante"]["kappa"], color=COL["ER κ constante"], ls=":", lw=1.5, label="κ constante")
ax.set_xlabel("Aceleración bariónica local g_b [m/s²]"); ax.set_ylabel("κ efectivo (puentes ER por masa bariónica)")
ax.set_title("Cuánto pesa la red ER según la gravedad local", loc="left"); ax.legend(fontsize=9)
plt.show()

## 8. Relación de aceleración radial (RAR) y diagnóstico de residuos

Arriba: cada punto de cada galaxia, con la aceleración observada $g_{obs} = V_{obs}^2/r$ frente a la bariónica $g_b$ (Υ = 0.5). Es la relación que cualquier explicación de la materia oscura tiene que reproducir.

Abajo: mediana del residuo $\log(g_{obs}/g_{modelo})$ por intervalos de $g_b$. Un modelo perfecto es una línea plana en 0. Así se ve **en qué régimen** falla cada modelo.

In [ ]:
xs, ys, res_m = [], [], {m: [] for m in ORDER if m != "Solo bariones"}
for name, g in GAL.items():
    x = np.clip(vb2(g, 0.5), 1e-6, None) / g["r"] * TO_MS2
    y = g["vobs"] ** 2 / g["r"] * TO_MS2
    ok = (g["vobs"] > 0)
    xs.append(x[ok]); ys.append(y[ok])
    for m in res_m:
        vm = np.clip(FITS[name][m]["v"], 1e-3, None)
        res_m[m].append(np.log10(g["vobs"][ok] ** 2 / vm[ok] ** 2))
xs, ys = np.concatenate(xs), np.concatenate(ys)
bins = np.linspace(-12.5, -8.5, 17); mid = 0.5 * (bins[1:] + bins[:-1])

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, gridspec_kw=dict(height_ratios=[2, 1.3], hspace=0.06))
a1.plot(np.log10(xs), np.log10(ys), ".", ms=2, color="#9a9a95", alpha=0.4, label="Puntos SPARC")
xx = np.linspace(-12.5, -8.5, 100); gbx = 10 ** xx
a1.plot(xx, np.log10(gbx / (1 - np.exp(-np.sqrt(gbx / 1.2e-10)))), color=COL["RAR McGaugh"], label="RAR de McGaugh")
a1.plot(xx, xx, ls="--", lw=1, color=COL["Solo bariones"], label="Solo bariones (1:1)")
a1.set_ylabel("log10 g_obs [m/s²]"); a1.legend(fontsize=9, loc="upper left")
a1.set_title("Relación de aceleración radial", loc="left")
for m in ["NFW ΛCDM (c–M)", "Verlinde", "RAR McGaugh", "ER κ constante", "ER no lineal κ(g)"]:
    rr = np.concatenate(res_m[m]); xb = np.log10(xs)
    med = [np.median(rr[(xb >= lo) & (xb < hi)]) if np.sum((xb >= lo) & (xb < hi)) > 10 else np.nan
           for lo, hi in zip(bins[:-1], bins[1:])]
    a2.plot(mid, med, "o-", ms=3.5, color=COL[m], label=m)
a2.axhline(0, color="#c3c2b7", lw=1)
a2.set_xlabel("log10 g_b [m/s²]"); a2.set_ylabel("log10 (g_obs / g_modelo)")
a2.legend(fontsize=8, ncol=2)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.linspace(-1, np.log10(1.5), 25)
for m in ["NFW ΛCDM (c–M)", "RAR McGaugh", "ER κ constante", "ER no lineal κ(g)"]:
    ax.hist(np.log10(RES.loc[RES.modelo == m, "ups"]), bins=bins, histtype="step", lw=2, color=COL[m], label=m)
ax.axvline(np.log10(0.5), color="#222222", lw=1, ls=":", label="Υ esperado ≈ 0.5")
ax.set_xlabel("log10 Υ (masa estelar por unidad de luz)"); ax.set_ylabel("Galaxias")
ax.set_title("¿Qué masa estelar pide cada modelo?", loc="left"); ax.legend(fontsize=8.5)
plt.show()

## 9. Prueba 2: validación cruzada entre galaxias (5 grupos)

Para cada variante de la red ER, se aprenden los parámetros globales con el 80 % de las galaxias y se predice el 20 % restante.

In [ ]:
names = np.array(list(GAL)); perm = RNG.permutation(len(names)); folds = np.array_split(perm, 5)
CVP = {m: {} for m in ER_MODELS}; cv_rows, fold_par = [], []
t0 = time.time()
for fi, idx in enumerate(folds):
    test = set(names[idx]); train = {k: v for k, v in GAL.items() if k not in test}
    for m in ER_MODELS:
        gf = global_er(train, m, verbose=False)
        fold_par.append(dict(grupo=fi, modelo=m, kappa=gf["kappa"], beta=gf["beta"], lam=gf["lam"]))
        for name in test:
            CVP[m][name] = (gf["kappa"], gf["beta"], gf["lam"])
            f = fit_er_one(GAL[name], gf["kappa"], gf["beta"], gf["lam"])
            cv_rows.append(dict(modelo=m, galaxia=name, fuera=f["chi2"] / max(f["n"] - 1, 1),
                                dentro=FITS[name][m]["chi2"] / max(FITS[name][m]["n"] - 1, 1)))
    print(f"  grupo {fi+1}/5 ({time.time()-t0:.0f} s)")
CV = pd.DataFrame(cv_rows); FP = pd.DataFrame(fold_par)
cv_sum = CV.groupby("modelo").agg(chi2r_dentro=("dentro", "median"), chi2r_fuera=("fuera", "median"))
cv_sum["cociente"] = cv_sum.chi2r_fuera / cv_sum.chi2r_dentro
print(FP.groupby("modelo")[["kappa", "beta", "lam"]].agg(["mean", "std"]).round(3).to_string())
cv_sum.loc[ER_MODELS].round(2)

## 10. Prueba 3: extrapolación radial (60 % interior → 40 % exterior)

Las variantes ER usan los parámetros globales del grupo de validación cruzada en el que esa galaxia **no** participó.

In [ ]:
EXT_MODELS = ["NFW libre", "NFW ΛCDM (c–M)", "Verlinde", "RAR McGaugh"] + ER_MODELS
ext_rows = []; t0 = time.time()
for name, g in GAL.items():
    N = len(g["r"])
    if N < 10:
        continue
    tr = np.zeros(N, bool); tr[: int(0.6 * N)] = True; te = ~tr
    for m in EXT_MODELS:
        if m in LS_MODELS:
            f = fit_ls(g, m, mask=tr, n_starts=3)
        else:
            f = fit_er_one(g, *CVP[m][name], mask=tr)
        ext_rows.append(dict(galaxia=name, modelo=m, interior=f["chi2"] / max(tr.sum() - f["k"], 1),
                             exterior=chi2_on(g, f["v"], te) / te.sum()))
EXT = pd.DataFrame(ext_rows)
ext_sum = EXT.groupby("modelo").agg(chi2r_interior=("interior", "median"), chi2r_exterior=("exterior", "median"))
ext_sum["empeora_x"] = ext_sum.chi2r_exterior / ext_sum.chi2r_interior
print(f"{EXT.galaxia.nunique()} galaxias, {time.time()-t0:.0f} s")
ext_sum.loc[EXT_MODELS].round(2)

## 11. NGC 3198, NGC 2403 y DDO 154

In [ ]:
SHOW = ["Solo bariones", "NFW ΛCDM (c–M)", "Verlinde", "RAR McGaugh", "ER κ constante", "ER no lineal κ(g)"]
def plot_detail(name):
    if name not in GAL:
        print(f"{name} no está en la muestra"); return
    g, F = GAL[name], FITS[name]
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(8.5, 6.8), sharex=True,
                                  gridspec_kw=dict(height_ratios=[3, 1.2], hspace=0.06))
    r = g["r"]
    ax.errorbar(r, g["vobs"], g["err"], fmt="o", ms=4, color=COL["Datos"], ecolor="#9a9a95", elinewidth=1,
                label="Observado", zorder=5)
    for m in SHOW:
        f = F[m]; c2 = f["chi2"] / max(f["n"] - f["k"], 1)
        ax.plot(r, f["v"], "--" if m == "Solo bariones" else "-", color=COL[m], label=f"{m} (χ²ν={c2:.1f})")
        if m != "Solo bariones":
            axr.plot(r, (g["vobs"] - f["v"]) / g["err"], "o-", ms=2.5, lw=1, color=COL[m])
    if len(r) >= 10:
        for a in (ax, axr):
            a.axvline(r[int(0.6 * len(r)) - 1], color="#9a9a95", ls=":", lw=1.2)
    ax.set_ylim(0, None); ax.set_ylabel("Velocidad [km/s]"); ax.set_title(name, loc="left")
    ax.legend(fontsize=8.5, loc="lower right")
    axr.axhline(0, color="#c3c2b7", lw=1); axr.set_ylabel("Residuo [σ]"); axr.set_xlabel("Radio [kpc]")
    plt.show()
    t = pd.DataFrame([dict(Modelo=m, chi2r=round(F[m]["chi2"] / max(F[m]["n"] - F[m]["k"], 1), 2),
                           Υ=round(F[m]["p"][0], 3)) for m in ORDER]).set_index("Modelo")
    e = EXT[EXT.galaxia == name].set_index("modelo")[["interior", "exterior"]].round(2)
    return t.join(e.rename(columns=lambda c: "extrap_" + c))

plot_detail("NGC3198")

In [ ]:
plot_detail("NGC2403")

In [ ]:
plot_detail("DDO154")

## 12. Veredicto automático

In [ ]:
def veredicto():
    S = SUMM.BIC_total; out = []
    if not HAVE_SPARC:
        out.append("⚠️  Datos SINTÉTICOS: este veredicto solo comprueba que el código funciona.")
    d = S["ER κ constante"] - S["ER no lineal κ(g)"]
    out.append(f"1) No lineal vs κ constante: ΔBIC = {d:+.0f} "
               + ("→ la dependencia con la aceleración MEJORA mucho el modelo." if d > 10 else
                  "→ no mejora de forma clara." if d > -10 else "→ empeora."))
    for rival in ["RAR McGaugh", "NFW ΛCDM (c–M)", "NFW libre"]:
        dd = S[rival] - S["ER no lineal κ(g)"]
        out.append(f"   vs {rival}: ΔBIC = {dd:+.0f} "
                   + ("(gana la red ER)" if dd > 10 else "(empate)" if dd > -10 else f"(gana {rival})"))
    out.append(f"2) β = {gnl['beta']:.2f} [{unl['beta'][0]:.2f}–{unl['beta'][1]:.2f}]: "
               + ("compatible con 1/2 (Tully–Fisher natural)." if unl["beta"][0] <= 0.5 <= unl["beta"][1]
                  else "no compatible con 1/2."))
    dOm = S["ER κ = Ωc/Ωb"] - S["ER κ constante"]
    out.append(f"3) κ = Ω_c/Ω_b frente a κ libre: ΔBIC = {dOm:+.0f} → "
               + ("la coincidencia cósmica sobrevive como predicción." if dOm <= 2 else
                  "fijar κ al valor cósmico empeora poco." if dOm <= 10 else
                  "fijar κ al valor cósmico empeora: con κ constante la coincidencia no basta."))
    if GLOB["ER no lineal κ(g)"]["edge"]:
        out.append("   Nota: el ajuste no lineal toca el borde de la rejilla (" + ", ".join(GLOB["ER no lineal κ(g)"]["edge"])
                   + "). Si es λ mínimo, el modelo prefiere ser LOCAL, como MOND/RAR.")
    for m in ER_MODELS:
        out.append(f"4) Validación cruzada {m}: fuera/dentro = {cv_sum.loc[m, 'cociente']:.2f}")
    ex = ext_sum.loc[EXT_MODELS, "chi2r_exterior"]
    out.append(f"5) Mejor predicción del exterior: {ex.idxmin()} (χ²ν = {ex.min():.2f}); "
               f"red ER no lineal: {ex['ER no lineal κ(g)']:.2f}; RAR: {ex['RAR McGaugh']:.2f}.")
    out.append(f"6) Υ mediana: ER no lineal {SUMM.loc['ER no lineal κ(g)', 'Υ_mediana']}, "
               f"ER κ constante {SUMM.loc['ER κ constante', 'Υ_mediana']} (esperado ≈ 0.5).")
    print("\n".join(out))

veredicto()

## 13. ¿Qué sería nuevo aquí? (léelo con los resultados delante)

- **Si la red no lineal empata con la RAR de McGaugh:** el éxito empírico es compartido. La RAR (y MOND) ya describen bien las curvas de rotación. Lo nuevo sería la **explicación**: entrelazamiento que sobrevive donde hay poca decoherencia, con una normalización ligada a Ω_c/Ω_b.
- **Lo que distingue a la red ER de MOND es λ.** La red es *no local*: la masa de la red en un radio depende de los bariones de su entorno, en un alcance λ. MOND/RAR es estrictamente local. Esto da una **predicción diferenciable**: cerca de rasgos bruscos de la distribución de gas o estrellas (bordes de disco, anillos, brazos), la red ER suaviza la señal a escala λ y la RAR no. Si λ sale bien acotado (no en el borde de la rejilla), esa es la prueba a diseñar.
- **Si β ≈ 1/2 sale solo de los datos,** es una consistencia notable, aunque esperable: es la misma estructura matemática que lleva a Tully–Fisher en MOND.
- **Siguiente frontera:** cúmulos de galaxias y lentes gravitacionales, donde MOND tiene problemas conocidos. Una red no local podría comportarse distinto.